# Lecture 8 · Notebook 4: Datasets and the Trainer

Natural Language Processing (2026), SeoulTech.

We fine-tune DistilBERT for movie-review sentiment (Rotten Tomatoes) with `datasets` and `Trainer`:
the pre-train → fine-tune recipe of Lecture 6, in about thirty lines.

*Runtime: use a GPU (Colab: Runtime → Change runtime type → T4 GPU). Training takes a few minutes.*

In [ ]:
# Run this cell first. Colab already has PyTorch; this installs the tested library versions.
%pip install -q "transformers>=5.20,<6" "datasets>=5.1,<6" "accelerate>=1.10"

import torch, transformers, datasets
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

## 1. Load a dataset from the Hub

In [ ]:
from datasets import load_dataset

raw = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(raw)
print(raw["train"].features)
print(raw["train"][0])

In [ ]:
labels = raw["train"].features["label"].names            # ['neg', 'pos']
import collections
print(collections.Counter(raw["train"]["label"]))
for ex in raw["train"].shuffle(seed=0).select(range(3)):
    print(labels[ex["label"]], "|", ex["text"])

A `Dataset` is stored in Apache Arrow format on disk and memory-mapped, so even large datasets
do not have to fit in RAM. Useful methods: `shuffle`, `select`, `filter`, `map`, `train_test_split`, `to_pandas`.

In [ ]:
lengths = raw["train"].map(lambda ex: {"n_words": len(ex["text"].split())})
print("mean words:", sum(lengths["n_words"]) / len(lengths))
short = raw["train"].filter(lambda ex: len(ex["text"].split()) < 8)
print(len(short), "reviews with fewer than 8 words, e.g.", short[0]["text"])

## 2. Tokenize with `map`

`batched=True` passes 1,000 examples at a time to the function, which is much faster than one at a time.
We do **not** pad here: padding happens per batch later (dynamic padding).

In [ ]:
from transformers import AutoTokenizer

ckpt = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(ckpt)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized = raw.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized)
print(tokenized["train"][0])

In [ ]:
from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer=tokenizer)
features = [tokenized["train"][i] for i in range(4)]
print([len(f["input_ids"]) for f in features])
batch = collator(features)
print({k: v.shape for k, v in batch.items()})          # padded to the longest of these four

## 3. The model

`AutoModelForSequenceClassification` loads the pre-trained DistilBERT body and adds a **new, randomly
initialized** classification head. The warning about newly initialized weights is expected.

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "neg", 1: "pos"}
model = AutoModelForSequenceClassification.from_pretrained(
    ckpt, num_labels=2, id2label=id2label, label2id={v: k for k, v in id2label.items()})
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

## 4. Metrics, arguments, and the Trainer

In [ ]:
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

In [ ]:
import torch
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="distilbert-rotten-tomatoes",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_steps=0.1,               # a float < 1 is a fraction of the training steps
    eval_strategy="epoch",          # evaluate on the validation set after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    fp16=torch.cuda.is_available(), # mixed precision on the GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

Accuracy **before** training: the head is random, so expect about 50%.

In [ ]:
print(trainer.evaluate())

In [ ]:
trainer.train()

In [ ]:
print(trainer.evaluate(tokenized["test"]))           # report the test set once, at the end

## 5. Predictions and errors

In [ ]:
pred = trainer.predict(tokenized["test"])
y_pred = pred.predictions.argmax(-1)
y_true = pred.label_ids
cm = np.zeros((2, 2), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1
print("rows = true (neg, pos), columns = predicted\n", cm)

wrong = np.where(y_pred != y_true)[0][:5]
for i in wrong:
    print(id2label[int(y_true[i])], "->", id2label[int(y_pred[i])], "|", raw["test"][int(i)]["text"])

## 6. Save the model and use it in a pipeline

In [ ]:
trainer.save_model("distilbert-rotten-tomatoes/best")
from transformers import pipeline
clf = pipeline("text-classification", model="distilbert-rotten-tomatoes/best", device=device)
clf(["An instant classic.", "Two hours I will never get back."])

To share the model, log in (`huggingface_hub.login()`) and call `trainer.push_to_hub()`;
the model card is written for you.

## Try it

1. Train with `learning_rate=5e-5` and with `2e-6`. How does validation accuracy change?
2. Train on only 500 examples (`tokenized["train"].shuffle(seed=0).select(range(500))`). How much accuracy do you lose?
3. Replace the checkpoint with `google-bert/bert-base-uncased`. Is the gain worth the longer training time?